# W13A2 — Important Python Libraries

| Library | Category | Purpose |
|---------|----------|---------|
| `os` | Standard Library | OS interface |
| `sys` | Standard Library | Interpreter control |
| `numpy` | Third-party | Numerical computing |
| `pathlib` | Standard Library | File path handling |
| `datetime` | Standard Library | Dates and times |
| `logging` | Standard Library | Application logging |
| `django` | Third-party | Full-stack web framework |
| `flask` | Third-party | Micro web framework |
| `fastapi` | Third-party | Async API framework |
| `bcrypt` | Third-party | Password hashing |
| `pytest` | Third-party | Testing framework |
| `pylint` | Third-party | Code linter |

---
## 6. `logging` — Application Logging

The standard way to record what a program does at runtime. Unlike `print()`, `logging` supports severity levels, multiple output destinations, and structured formatting — without changing application code to enable or silence output.

**Log levels (low → high):** `DEBUG` → `INFO` → `WARNING` → `ERROR` → `CRITICAL`

**Core components:**
- **Logger** — the object you call (`logging.getLogger(name)`)
- **Handler** — where output goes (`StreamHandler` for console, `FileHandler` for files)
- **Formatter** — controls the output layout (timestamp, level, message)

In [ ]:
import logging

# Example 1: Basic configuration — emit messages at each severity level
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)-8s] %(message)s",
    datefmt="%H:%M:%S",
)

logging.debug("Connecting to database...")
logging.info("Server started on port 8080")
logging.warning("Config file not found; using defaults")
logging.error("Failed to read user record id=42")
logging.critical("Out of disk space — shutting down")

In [ ]:
import logging
from pathlib import Path

# Example 2: Named logger writing to both console and a file
logger = logging.getLogger("myapp.auth")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    fmt = logging.Formatter("%(asctime)s [%(name)s] [%(levelname)-8s] %(message)s", datefmt="%H:%M:%S")

    console = logging.StreamHandler()
    console.setLevel(logging.INFO)      # INFO and above to console
    console.setFormatter(fmt)

    log_file = logging.FileHandler("/tmp/myapp.log", mode="w", encoding="utf-8")
    log_file.setLevel(logging.DEBUG)    # all levels to file
    log_file.setFormatter(fmt)

    logger.addHandler(console)
    logger.addHandler(log_file)

logger.debug("Auth module initialised")
logger.info("User 'alice' login attempt")
logger.warning("Password expires in 3 days")
logger.info("User 'alice' authenticated successfully")

print("\n--- /tmp/myapp.log ---")
print(Path("/tmp/myapp.log").read_text(encoding="utf-8"))

---
## 9. `fastapi` — Modern Async API Framework

Built on Starlette (ASGI) and Pydantic, FastAPI combines Python type hints with automatic request validation and OpenAPI documentation generation. `async def` route handlers enable high concurrency. Visiting `/docs` gives an interactive Swagger UI with no extra setup.

**Advantages over Flask:** automatic validation, auto-generated docs, native async support.

In [ ]:
# Example 1: Typed request/response with Pydantic validation
# Run with:  uvicorn app:app --reload
# Docs at:   http://127.0.0.1:8000/docs

"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, EmailStr

app = FastAPI(title="User Service")

class UserCreate(BaseModel):
    name:  str
    email: EmailStr
    age:   int

class UserResponse(BaseModel):
    id:    int
    name:  str
    email: str

USERS: list[UserResponse] = []

@app.post("/users", response_model=UserResponse, status_code=201)
def create_user(payload: UserCreate):
    user = UserResponse(id=len(USERS) + 1, name=payload.name, email=payload.email)
    USERS.append(user)
    return user

@app.get("/users/{user_id}", response_model=UserResponse)
def get_user(user_id: int):
    for u in USERS:
        if u.id == user_id:
            return u
    raise HTTPException(status_code=404, detail="User not found")
"""

print("FastAPI typed-body example (save as app.py and run with uvicorn).")

In [ ]:
# Example 2: Async route with query parameters

"""
from fastapi import FastAPI, Query
import asyncio

app = FastAPI()

async def fetch_from_db(item_id: int) -> dict:
    await asyncio.sleep(0.01)   # simulate async I/O
    return {"id": item_id, "value": f"item_{item_id}"}

@app.get("/items")
async def list_items(
    skip:  int = Query(0,  ge=0),
    limit: int = Query(10, ge=1, le=100),
):
    items = [await fetch_from_db(i) for i in range(skip, skip + limit)]
    return {"skip": skip, "limit": limit, "items": items}
"""

print("FastAPI async route example (save as app.py and run with uvicorn).")